In [ ]:
"""
==================================================================
INTESA SANPAOLO (ISP) - EQUITY RESEARCH / DCF VALUATION MODEL
Prepared by: Ali Akbar
Date: 2026-05-21
Methodology: DCF(3-scenarion) + compareable companies (comps) + Risk analysis
Data Sources: yfinance, company financial statements, registro imprese
===================================================================

ANALYST NOTES:
- All figures in EUR unless otherwise stated.
- DCF valuation based on three scenarios: Base, Bull, and Bear.
- Tax rate assumed at 27% based on ISP's historical effective tax rate.
- Net Interest Margin (NIM): evolution with the rate cycle
- Cost to income ratio (CIR): evolution with the tax cycle
- Return on tangible equity (ROTE): Profitability for shareholders

====================================================================
"""
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from IPython.display import display
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ______Data extraction____________________________________________
ticker = 'ISP.MI'

stock = yf.Ticker(ticker)
financials    = stock.financials
balance_sheet = stock.balance_sheet
cashflow      = stock.cashflow
info          = stock.info

df_fi = pd.DataFrame(financials)
df_bs = pd.DataFrame(balance_sheet)
df_cf = pd.DataFrame(cashflow)

# Extracting key information from income statement
df1 = pd.DataFrame({
    'Total_Revenue': financials.loc['Total Revenue'],
    'Net_Income': financials.loc['Net Income'],
    'operating_expenses': financials.loc['Operating Expense'],

}).dropna()

# Extracting key information from balance sheet
df2 = pd.DataFrame({
    'total_debt': balance_sheet.loc['Total Debt'],
    'cash': balance_sheet.loc['Cash And Cash Equivalents'],
    'tangible_bv': balance_sheet.loc['Tangible Book Value']
}).dropna()

#Extracting key information from cash flow statement
df3 = pd.DataFrame({
    'OCF': cashflow.loc['Operating Cash Flow'],
    'FCF': cashflow.loc['Free Cash Flow'],
    'Depreciation': cashflow.loc['Depreciation And Amortization'],

}).dropna()

# Calculating EBIT 
EBIT = df1['Total_Revenue'] - df1['operating_expenses']

# ______calculating key kpi _____________________________________________________

Net_Income_Margin = df1['Net_Income'] / df1['Total_Revenue']

# Calculating Net interest margin (NIM) = Net interest income / Average earning assets
# Getting Average earning assets from balance sheet is not straightforward, so we will use Total Assets as a proxy
avg_assets = (df_bs.loc['Total Assets'] + df_bs.loc['Total Assets'].shift(-1)) / 2

NIM = df_fi.loc['Net Interest Income'] / avg_assets  # NIM = NII / Avg Total Assets

# Cost to income ratio (CIR) = Operating expenses / Total revenue
CIR = df1['operating_expenses'] / df1['Total_Revenue']

# Return on tangible equity (ROTE) = Net income / Tangible book value
ROTE = df1['Net_Income'] / df2['tangible_bv']

# ______Formatting helper__________________________________________
def fmt_value(v):
    """Convert a raw number to a human-readable string (B / M / K)."""
    if pd.isna(v):
        return "N/A"
    try:
        v = float(v)
    except (TypeError, ValueError):
        return str(v)
    sign = "-" if v < 0 else ""
    v = abs(v)
    if v >= 1e12:
        return f"{sign}{v/1e12:.2f}T"
    elif v >= 1e9:
        return f"{sign}{v/1e9:.2f}B"
    elif v >= 1e6:
        return f"{sign}{v/1e6:.2f}M"
    elif v >= 1e3:
        return f"{sign}{v/1e3:.2f}K"
    else:
        return f"{sign}{v:.4f}"

def format_df(df):
    """Apply fmt_value to every cell; keep column names as year strings."""
    formatted = df.copy().astype(object)
    for col in formatted.columns:
        formatted[col] = formatted[col].apply(fmt_value)
    formatted.columns = [str(c)[:10] for c in formatted.columns]  # trim timestamps to date
    return formatted

df_fi_fmt = format_df(df_fi)
df_bs_fmt = format_df(df_bs)
df_cf_fmt = format_df(df_cf)

# ______Display____________________________________________________
print("=" * 60)
print("INCOME STATEMENT  (EUR)")
print("=" * 60)
display(df_fi_fmt)

print("\n" + "=" * 60)
print("BALANCE SHEET  (EUR)")
print("=" * 60)
display(df_bs_fmt)

print("\n" + "=" * 60)
print("CASH FLOW STATEMENT  (EUR)")
print("=" * 60)
display(df_cf_fmt)

df_fi_fmt1 = format_df(df1)
df_bs_fmt2 = format_df(df2)
df_cf_fmt3 = format_df(df3)
print("\n"+'=' * 60)
print("KEY METRICS")
print('=' * 60)
print("Income statement key metrics:")
display(df_fi_fmt1)

print("\n" + "=" * 60)
print("Balance sheet key metrics:")
display(df_bs_fmt2)

print("\n" + "=" * 60)
print("Cash flow statement key metrics:")
display(df_cf_fmt3)


In [ ]:
# ============================================================
# KPI DASHBOARD  —  INTESA SANPAOLO (ISP.MI)
# Standalone: requires only df_fi, df_bs, df_cf from Cell 1
# ============================================================
import numpy as np

def _safe_loc(df, key):
    return df.loc[key] if key in df.index else pd.Series(dtype=float)

def _fmt(v, mode='eur'):
    if v is None: return "  N/A"
    try:    v = float(v)
    except: return "  N/A"
    if pd.isna(v): return "  N/A"
    sign = "-" if v < 0 else ""
    av   = abs(v)
    if mode == 'pct': return f"{v*100:+.2f}%"
    if mode == 'x':   return f"{v:.2f}x"
    if av >= 1e12:    return f"{sign}EUR {av/1e12:.2f}T"
    if av >= 1e9:     return f"{sign}EUR {av/1e9:.2f}B"
    if av >= 1e6:     return f"{sign}EUR {av/1e6:.2f}M"
    if av >= 1e3:     return f"{sign}EUR {av/1e3:.2f}K"
    return f"{sign}EUR {av:.4f}"

SEP  = "=" * 72
SEP2 = "-" * 72

# ---- key series ------------------------------------------------
total_revenue       = _safe_loc(df_fi, 'Total Revenue')
net_income          = _safe_loc(df_fi, 'Net Income')
op_expense          = _safe_loc(df_fi, 'Operating Expense')
net_interest_income = _safe_loc(df_fi, 'Net Interest Income')
gross_profit        = _safe_loc(df_fi, 'Gross Profit')
total_assets        = _safe_loc(df_bs, 'Total Assets')
tangible_bv         = _safe_loc(df_bs, 'Tangible Book Value')
stockholders_eq     = _safe_loc(df_bs, 'Stockholders Equity')
total_debt          = _safe_loc(df_bs, 'Total Debt')
cash                = _safe_loc(df_bs, 'Cash And Cash Equivalents')
ocf                 = _safe_loc(df_cf, 'Operating Cash Flow')
fcf                 = _safe_loc(df_cf, 'Free Cash Flow')
capex               = _safe_loc(df_cf, 'Capital Expenditure')
da                  = _safe_loc(df_cf, 'Depreciation And Amortization')

# ---- corrected KPI formulas ------------------------------------
avg_assets  = (total_assets + total_assets.shift(-1)) / 2
ebit        = total_revenue - op_expense
ni_margin   = net_income / total_revenue
cir         = op_expense / total_revenue
nim_bank    = net_interest_income / avg_assets
rota        = net_income / avg_assets
rote        = net_income / tangible_bv
roe         = net_income / stockholders_eq
leverage    = total_assets / stockholders_eq
debt_equity = total_debt / tangible_bv
net_debt    = total_debt - cash

cols = sorted(df_fi.columns, reverse=True)
yrs  = [str(c)[:4] for c in cols]

SECTIONS = [
    ("INCOME STATEMENT  (EUR)", [
        ("Total Revenue",                 total_revenue,       'eur'),
        ("Gross Profit",                  gross_profit,        'eur'),
        ("EBIT",                          ebit,                'eur'),
        ("Net Interest Income",           net_interest_income, 'eur'),
        ("Net Income",                    net_income,          'eur'),
    ]),
    ("CASH FLOW  (EUR)", [
        ("Operating Cash Flow",           ocf,   'eur'),
        ("Capital Expenditure",           capex, 'eur'),
        ("Free Cash Flow",                fcf,   'eur'),
        ("Depreciation & Amortization",   da,    'eur'),
    ]),
    ("PROFITABILITY & EFFICIENCY", [
        ("Net Income Margin",              ni_margin, 'pct'),
        ("NIM  (NII / Avg Total Assets)", nim_bank,  'pct'),
        ("Cost-to-Income  (CIR)",         cir,       'pct'),
        ("ROTA  (NI / Avg Total Assets)", rota,      'pct'),
        ("ROTE  (NI / Tangible BV)",      rote,      'pct'),
        ("ROE   (NI / Equity)",           roe,       'pct'),
    ]),
    ("BALANCE SHEET  (EUR)", [
        ("Total Assets",                  total_assets,    'eur'),
        ("Total Debt",                    total_debt,      'eur'),
        ("Cash & Equivalents",            cash,            'eur'),
        ("Net Debt  (Debt - Cash)",       net_debt,        'eur'),
        ("Tangible Book Value",           tangible_bv,     'eur'),
        ("Stockholders Equity",           stockholders_eq, 'eur'),
        ("Debt / Tangible Equity",        debt_equity,     'x'),
        ("Asset Leverage",                leverage,        'x'),
    ]),
]

LW = 34
CW = 15

# ---- display ---------------------------------------------------
print(SEP)
print(f"  KPI DASHBOARD  |  INTESA SANPAOLO (ISP.MI)  |  Figures in EUR")
print(SEP)

kpi_data = {}
for title, metrics in SECTIONS:
    print(f"\n  {title}")
    print(f"  {SEP2[:LW + CW*len(yrs)]}")
    print(f"  {'Metric':<{LW}}" + "".join(f"{yr:>{CW}}" for yr in yrs))
    print(f"  {'-'*(LW + CW*len(yrs))}")
    for label, series, mode in metrics:
        row = f"  {label:<{LW}}"
        kpi_data.setdefault(label, {})
        for col, yr in zip(cols, yrs):
            val = series.get(col) if len(series) > 0 else np.nan
            fv  = _fmt(val, mode)
            row += f"{fv:>{CW}}"
            kpi_data[label][yr] = fv
        print(row)

print(f"\n{SEP}")
print(f"  Source: yfinance (ISP.MI)  |  Analyst: Ali Akbar  |  Date: 2026-05-21")
print(f"  NIM = Net Interest Income / Avg Total Assets  (bank standard)")
print(f"  ROTA = Net Income / Avg Total Assets  |  ROTE = Net Income / Tangible BV")
print(SEP)

# ---- build df_kpi for Excel ------------------------------------
df_kpi = pd.DataFrame(kpi_data).T
df_kpi.index.name = "KPI / Metric"


In [ ]:
# Export all sheets to Excel
# NOTE: close the file in Excel first if it is already open, otherwise you get a PermissionError
output_path = "ISP_Analysis.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    df_fi_fmt.to_excel(writer, sheet_name="Income Statement")
    df_bs_fmt.to_excel(writer, sheet_name="Balance Sheet")
    df_cf_fmt.to_excel(writer, sheet_name="Cash Flow")
    df_kpi.to_excel(writer, sheet_name="KPI Dashboard")

print(f"Exported successfully -> {output_path}")
print(f"  Sheet 1 - Income Statement : {df_fi_fmt.shape[0]} rows x {df_fi_fmt.shape[1]} cols")
print(f"  Sheet 2 - Balance Sheet    : {df_bs_fmt.shape[0]} rows x {df_bs_fmt.shape[1]} cols")
print(f"  Sheet 3 - Cash Flow        : {df_cf_fmt.shape[0]} rows x {df_cf_fmt.shape[1]} cols")
print(f"  Sheet 4 - KPI Dashboard    : {df_kpi.shape[0]} metrics x {df_kpi.shape[1]} years")


In [ ]:
# ============================================================
# STEP 5 — FORWARD FINANCIAL MODEL  |  ISP.MI
# Projections: 2026 / 2027 / 2028   (3-year horizon)
# Scenarios  : Base | Bull | Bear
# Tax Rate   : 27%  |  Analyst: Ali Akbar
# Sources    : yfinance actuals (Cell 1), ISP Piano d'Impresa 2025-2028,
#              ECB rate path consensus (May 2026), sector benchmarks
# ============================================================

import matplotlib.ticker as mticker

# ______Anchor points (2025 actuals from Cell 1)___________________
base_nii        = 17.31e9   # Net Interest Income 2025 (yfinance)
base_fees       = 9.80e9    # Net fee & commission income
base_assets     = 959.89e9  # Total assets 2025 (yfinance)
base_loans      = 500.00e9  # Gross loans ~52% of assets
base_nim        = base_nii / base_assets
base_net_income = 9.32e9    # Net income 2025 (yfinance)
shares          = 17.39e9   # Shares outstanding 2025
tangible_bv     = 55.22e9   # Tangible book value 2025
rwa             = 310.00e9  # Risk-weighted assets
cet1_capital    = rwa * 0.135

TAX_RATE     = 0.27
PAYOUT_RATIO = 0.70
YEARS        = [2026, 2027, 2028]

scenarios = {
    'Base': {
        'nim_bps'    : [-10, -8, -5],
        'loan_growth': [0.02, 0.02, 0.02],
        'fee_growth' : [0.06, 0.06, 0.05],
        'cir'        : [0.44, 0.43, 0.42],
        'cor_bps'    : [28, 30, 30],
    },
    'Bull': {
        'nim_bps'    : [-5, -3, -2],
        'loan_growth': [0.04, 0.04, 0.03],
        'fee_growth' : [0.09, 0.08, 0.07],
        'cir'        : [0.41, 0.40, 0.39],
        'cor_bps'    : [20, 20, 18],
    },
    'Bear': {
        'nim_bps'    : [-20, -15, -10],
        'loan_growth': [0.00, 0.01, 0.01],
        'fee_growth' : [0.01, 0.02, 0.03],
        'cir'        : [0.47, 0.46, 0.45],
        'cor_bps'    : [55, 60, 55],
    },
}

results = {}
for sc_name, p in scenarios.items():
    rows = []
    nim = base_nim; loans = base_loans; fees = base_fees
    for i, yr in enumerate(YEARS):
        nim    = nim + p['nim_bps'][i] * 1e-4
        loans  = loans * (1 + p['loan_growth'][i])
        fees   = fees  * (1 + p['fee_growth'][i])
        assets = loans / 0.52
        nii    = assets * nim
        total_rev = nii + fees
        op_costs  = total_rev * p['cir'][i]
        llp       = loans * p['cor_bps'][i] * 1e-4
        pretax    = total_rev - op_costs - llp
        net_income = pretax * (1 - TAX_RATE)
        eps   = net_income / shares
        dps   = eps * PAYOUT_RATIO
        tbvps = tangible_bv / shares
        rote_fwd = net_income / tangible_bv
        cet1_r   = cet1_capital / rwa
        rows.append({
            'Year': yr,
            'NII (EUR B)': round(nii/1e9, 2),
            'Fee Income (EUR B)': round(fees/1e9, 2),
            'Total Revenue (EUR B)': round(total_rev/1e9, 2),
            'Op Costs (EUR B)': round(op_costs/1e9, 2),
            'LLP (EUR B)': round(llp/1e9, 2),
            'Net Income (EUR B)': round(net_income/1e9, 2),
            'EPS (EUR)': round(eps, 3),
            'DPS (EUR)': round(dps, 3),
            'ROTE (%)': round(rote_fwd * 100, 1),
            'CET1 (%)': round(cet1_r * 100, 1),
            'NIM (%)': round(nim * 100, 2),
        })
    df_sc = pd.DataFrame(rows).set_index('Year')
    results[sc_name] = df_sc

# Print results
SEP = '=' * 72
for sc, df in results.items():
    print(f'\n{SEP}')
    print(f'  {sc.upper()} SCENARIO')
    print(SEP)
    print(df.to_string())

# __ Forward model chart __________________________________________
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('ISP.MI — Forward Model 2026-2028  |  Base / Bull / Bear', fontsize=14, fontweight='bold')
colors = {'Base': '#1f77b4', 'Bull': '#2ca02c', 'Bear': '#d62728'}
metrics = ['Net Income (EUR B)', 'EPS (EUR)', 'DPS (EUR)']
for ax, metric in zip(axes, metrics):
    for sc, df in results.items():
        ax.plot(df.index, df[metric], 'o-', color=colors[sc], label=sc, lw=2, ms=6)
    ax.set_title(metric); ax.legend(); ax.grid(True, alpha=0.3)
    ax.set_xticks(YEARS)
plt.tight_layout()
plt.savefig('ISP_forward_model.png', dpi=120, bbox_inches='tight')
plt.show()
print('\nForward model chart saved.')


In [ ]:
# ============================================================
# STEP 5 — EXPORT FORWARD MODEL TO EXCEL
# Adds 'Forward Model' sheet to existing ISP_Analysis.xlsx
# ============================================================

from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

output_path = 'ISP_Analysis.xlsx'
sheet_name  = 'Forward Model'

wb = load_workbook(output_path)
if sheet_name in wb.sheetnames:
    del wb[sheet_name]
ws = wb.create_sheet(sheet_name)

GREEN      = '007236'
DARK       = '1A1A2E'
LIGHT_GREY = 'F2F2F2'
WHITE      = 'FFFFFF'

def header_cell(ws, row, col, value, bg=GREEN, fg=WHITE, bold=True):
    c = ws.cell(row=row, column=col, value=value)
    c.fill      = PatternFill('solid', fgColor=bg)
    c.font      = Font(bold=bold, color=fg, size=10)
    c.alignment = Alignment(horizontal='center', vertical='center')
    return c

def data_cell(ws, row, col, value, bg=WHITE, bold=False):
    c = ws.cell(row=row, column=col, value=value)
    c.fill      = PatternFill('solid', fgColor=bg)
    c.font      = Font(bold=bold, size=10)
    c.alignment = Alignment(horizontal='center', vertical='center')
    return c

def thin_border():
    s = Side(style='thin', color='CCCCCC')
    return Border(left=s, right=s, top=s, bottom=s)

scenario_colors = {'Base': '1F77B4', 'Bull': '2CA02C', 'Bear': 'D62728'}
current_row = 1

ws.merge_cells(start_row=current_row, start_column=1, end_row=current_row, end_column=13)
title = ws.cell(row=current_row, column=1,
                value='ISP.MI — Forward Financial Model 2026-2028  |  Base / Bull / Bear')
title.font      = Font(bold=True, size=13, color=WHITE)
title.fill      = PatternFill('solid', fgColor=DARK)
title.alignment = Alignment(horizontal='center', vertical='center')
ws.row_dimensions[current_row].height = 28
current_row += 2

for scenario_name, df in results.items():
    sc_color = scenario_colors[scenario_name]
    ws.merge_cells(start_row=current_row, start_column=1,
                   end_row=current_row, end_column=len(df.columns) + 1)
    sc_header = ws.cell(row=current_row, column=1, value=f'  {scenario_name.upper()} SCENARIO')
    sc_header.fill      = PatternFill('solid', fgColor=sc_color)
    sc_header.font      = Font(bold=True, size=11, color=WHITE)
    sc_header.alignment = Alignment(horizontal='left', vertical='center')
    ws.row_dimensions[current_row].height = 22
    current_row += 1

    header_cell(ws, current_row, 1, 'Year', bg='333333')
    for ci, col_name in enumerate(df.columns, 2):
        header_cell(ws, current_row, ci, col_name, bg='333333')
    current_row += 1

    for yr, row_data in df.iterrows():
        data_cell(ws, current_row, 1, yr, bold=True)
        for ci, val in enumerate(row_data, 2):
            bg = LIGHT_GREY if current_row % 2 == 0 else WHITE
            data_cell(ws, current_row, ci, val, bg=bg)
        current_row += 1
    current_row += 1

for col in range(1, 14):
    ws.column_dimensions[get_column_letter(col)].width = 18

wb.save(output_path)
print(f'Forward Model sheet saved to {output_path}')


In [ ]:
# ============================================================
# STEP 6 — MULTI-METHOD VALUATION  |  ISP.MI
# Methods: P/BV | P/E | DDM | Excess Return | SOTP | Peer Comps
# Analyst: Ali Akbar
# ============================================================

import matplotlib.ticker as mticker

current_price = info.get('currentPrice', info.get('regularMarketPrice', 4.52))
print(f'ISP.MI current price : EUR {current_price:.2f}\n')

tbv        = float(df_bs.loc['Tangible Book Value'].iloc[0])
shares_out = float(df_bs.loc['Ordinary Shares Number'].iloc[0])
net_inc    = float(df_fi.loc['Net Income'].iloc[0])

tbvps     = tbv / shares_out
eps_trail = net_inc / shares_out
eps_fwd   = 0.577
dps_fwd   = 0.404
rote      = net_inc / tbv

rf   = 0.038
beta = 1.15
erp  = 0.055
ke   = rf + beta * erp
g    = 0.025

print('=' * 60)
print('  KEY VALUATION INPUTS')
print('=' * 60)
print(f'  TBVPS              : EUR {tbvps:.2f}')
print(f'  EPS 2025 trailing  : EUR {eps_trail:.3f}')
print(f'  EPS 2026E (Base)   : EUR {eps_fwd:.3f}')
print(f'  DPS 2026E (Base)   : EUR {dps_fwd:.3f}')
print(f'  ROTE               : {rote*100:.1f}%')
print(f'  Cost of Equity Ke  : {ke*100:.1f}%')
print(f'  Terminal growth g  : {g*100:.1f}%')

pbv_low  = tbvps * 1.20
pbv_base = tbvps * 1.40
pbv_high = tbvps * 1.60

pe_low  = eps_fwd * 8.5
pe_base = eps_fwd * 10.0
pe_high = eps_fwd * 11.5

ddm_base = dps_fwd / (ke - g)
ddm_low  = dps_fwd / ((ke + 0.005) - g)
ddm_high = dps_fwd / ((ke - 0.005) - g)

xs_ps       = (rote - ke) * tbvps
pv_explicit = sum(xs_ps / (1 + ke)**t for t in range(1, 4))
pv_terminal = (xs_ps / (ke - g)) / (1 + ke)**3
er_base     = tbvps + pv_explicit + pv_terminal
er_low      = er_base * 0.88
er_high     = er_base * 1.12

divisions = {
    'Banca dei Territori'          : (0.38, 0.90),
    'Corporate & Investment Banking': (0.20, 1.20),
    'International Subsidiaries'   : (0.15, 1.00),
    'Private Banking'              : (0.10, 2.00),
    'Asset Mgmt (Eurizon)'         : (0.10, 3.50),
    'Insurance'                    : (0.07, 1.20),
}
sotp_base = sum(p * m * tbvps for p, m in divisions.values())
sotp_low  = sotp_base * 0.90
sotp_high = sotp_base * 1.10

peers = {
    'UCG.MI' : {'pbv': 1.12, 'pe': 8.9},
    'BNP.PA' : {'pbv': 0.65, 'pe': 7.4},
    'SAN.MC' : {'pbv': 0.82, 'pe': 6.9},
    'INGA.AS': {'pbv': 0.89, 'pe': 8.1},
    'GLE.PA' : {'pbv': 0.55, 'pe': 5.8},
}
peer_pbv_list = [v['pbv'] for v in peers.values()]
peer_pe_list  = [v['pe']  for v in peers.values()]
comp_base = (np.median(peer_pbv_list) * tbvps + np.median(peer_pe_list) * eps_fwd) / 2
comp_low  = comp_base * 0.90
comp_high = comp_base * 1.10

valuation = {
    'P/BV'        : (pbv_low,  pbv_base,  pbv_high),
    'P/E'         : (pe_low,   pe_base,   pe_high),
    'DDM'         : (ddm_low,  ddm_base,  ddm_high),
    'Excess Return': (er_low,  er_base,   er_high),
    'SOTP'        : (sotp_low, sotp_base, sotp_high),
    'Peer Comps'  : (comp_low, comp_base, comp_high),
}

print('\n' + '=' * 60)
print('  VALUATION SUMMARY')
print('=' * 60)
print(f'  {"Method":<16} {"Bear":>7} {"Base":>7} {"Bull":>7}')
print(f'  {"-"*55}')
for method, (lo, base, hi) in valuation.items():
    print(f'  {method:<16} {lo:>6.2f} {base:>7.2f} {hi:>7.2f}')
pt_base = round(np.mean([v[1] for v in valuation.values()]), 2)
pt_bull = round(max(v[2] for v in valuation.values()), 2)
pt_bear = round(min(v[0] for v in valuation.values()), 2)
print(f'\n  Target Price (Base) : EUR {pt_base}')
print(f'  Target Price (Bull) : EUR {pt_bull}')
print(f'  Target Price (Bear) : EUR {pt_bear}')
print(f'  Current Price       : EUR {current_price:.2f}')
print(f'  Upside (Base)       : {(pt_base-current_price)/current_price*100:+.1f}%')

# Football field chart
methods = list(valuation.keys())
lows    = [v[0] for v in valuation.values()]
bases   = [v[1] for v in valuation.values()]
highs   = [v[2] for v in valuation.values()]
colors  = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd','#8c564b']

fig, ax = plt.subplots(figsize=(12, 6))
for i, (lo, base, hi, col) in enumerate(zip(lows, bases, highs, colors)):
    ax.barh(i, hi - lo, left=lo, height=0.5, color=col, alpha=0.4)
    ax.plot(base, i, 'd', color=col, ms=10, zorder=5)
    ax.text(lo - 0.05, i, f'{lo:.2f}', va='center', ha='right', fontsize=9)
    ax.text(hi + 0.05, i, f'{hi:.2f}', va='center', ha='left',  fontsize=9)
ax.axvline(current_price, color='red',   lw=2, ls='--', label=f'Current EUR {current_price:.2f}')
ax.axvline(pt_base,       color='green', lw=2, ls='-',  label=f'Target  EUR {pt_base}')
ax.set_yticks(range(len(methods)))
ax.set_yticklabels(methods)
ax.set_xlabel('Share Price (EUR)')
ax.set_title(f'ISP.MI — Valuation Football Field  |  Target: EUR {pt_base}  |  Rating: HOLD')
ax.legend(); ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('ISP_valuation_football_field.png', dpi=120, bbox_inches='tight')
plt.show()
print('\nValuation football field chart saved.')


In [ ]:
# ============================================================
# STEP 6 — EXPORT VALUATION TO EXCEL
# Adds 'Valuation' sheet to ISP_Analysis.xlsx
# ============================================================

from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

output_path = 'ISP_Analysis.xlsx'
sheet_name  = 'Valuation'

wb = load_workbook(output_path)
if sheet_name in wb.sheetnames:
    del wb[sheet_name]
ws = wb.create_sheet(sheet_name)

DARK  = '1A1A2E'; WHITE = 'FFFFFF'; GREEN = '007236'
LIGHT = 'F2F2F2'; GREY  = 'DDDDDD'

def hdr(row, col, val, bg=GREEN):
    c = ws.cell(row=row, column=col, value=val)
    c.fill = PatternFill('solid', fgColor=bg)
    c.font = Font(bold=True, color=WHITE, size=10)
    c.alignment = Alignment(horizontal='center')

def dat(row, col, val, bg=WHITE, bold=False):
    c = ws.cell(row=row, column=col, value=val)
    c.fill = PatternFill('solid', fgColor=bg)
    c.font = Font(bold=bold, size=10)
    c.alignment = Alignment(horizontal='center')

# Title
ws.merge_cells('A1:E1')
t = ws.cell(row=1, column=1, value='ISP.MI — Multi-Method Valuation  |  Analyst: Ali Akbar  |  May 2026')
t.fill = PatternFill('solid', fgColor=DARK)
t.font = Font(bold=True, size=13, color=WHITE)
t.alignment = Alignment(horizontal='center')
ws.row_dimensions[1].height = 28

# Valuation table
for ci, h in enumerate(['Method', 'Bear', 'Base', 'Bull', 'Spread'], 1):
    hdr(3, ci, h)
for ri, (method, (lo, base, hi)) in enumerate(valuation.items(), 4):
    bg = LIGHT if ri % 2 == 0 else WHITE
    dat(ri, 1, method, bg=bg, bold=True)
    dat(ri, 2, round(lo,   2), bg=bg)
    dat(ri, 3, round(base, 2), bg=bg)
    dat(ri, 4, round(hi,   2), bg=bg)
    dat(ri, 5, round(hi-lo, 2), bg=bg)

# Summary
row = 4 + len(valuation) + 1
ws.merge_cells(start_row=row, start_column=1, end_row=row, end_column=5)
s = ws.cell(row=row, column=1,
    value=f'Target Price (Base): EUR {pt_base}  |  Bull: EUR {pt_bull}  |  Bear: EUR {pt_bear}  |  Rating: HOLD')
s.fill = PatternFill('solid', fgColor='007236')
s.font = Font(bold=True, color=WHITE, size=11)
s.alignment = Alignment(horizontal='center')

# SOTP breakdown
row += 2
hdr(row, 1, 'SOTP Division', bg='333333')
hdr(row, 2, '% TBV', bg='333333')
hdr(row, 3, 'P/BV', bg='333333')
hdr(row, 4, 'Contribution', bg='333333')
row += 1
for ri, (div, (pct, mult)) in enumerate(divisions.items()):
    bg = LIGHT if ri % 2 == 0 else WHITE
    dat(row, 1, div, bg=bg, bold=True)
    dat(row, 2, f'{pct*100:.0f}%', bg=bg)
    dat(row, 3, f'{mult:.2f}x', bg=bg)
    dat(row, 4, round(pct * mult * tbvps, 2), bg=bg)
    row += 1

for col in range(1, 6):
    ws.column_dimensions[get_column_letter(col)].width = 22

wb.save(output_path)
print(f'Valuation sheet saved to {output_path}')


In [ ]:
# ============================================================
# STEP 8 — RISK REGISTER  |  ISP.MI
# Analyst: Ali Akbar
# ============================================================

from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

risks = [
    ('Macro',        'NIM Compression',         'ECB cuts faster than expected — NIM falls below 1.50% by 2027',          'High',   'High',   'Variable-rate repricing + fee income diversification'),
    ('Macro',        'Italian Recession',        'GDP contraction reduces loan demand and lifts NPLs',                     'Medium', 'High',   'Diversified in 40+ countries; CET1 13.5% absorbs shock'),
    ('Credit',       'NPL Cycle Turn',           'Employment -0.1% trend leads NPL deterioration by 6-12 months',         'Medium', 'High',   'Conservative 28bps CoR; coverage ratio >70%'),
    ('Credit',       'Alexbank / Egypt',         'EGP weakness from Red Sea disruption hits Egypt subsidiary P&L',         'Medium', 'Medium', 'Alexbank ~2% of consolidated assets; FX hedges in place'),
    ('Regulatory',   'Basel IV / DORA',          'New capital and operational resilience rules raise compliance costs',    'High',   'Medium', 'CET1 13.5% above Basel IV floor; IT investment ongoing'),
    ('Regulatory',   'ECB Supervisory Action',   'Targeted review of credit models or capital add-ons',                   'Low',    'High',   'Strong SREP track record; transparent disclosures'),
    ('Geopolitical', 'Middle East Escalation',   'Extended conflict keeps energy prices high — inflation sticky',         'Medium', 'Medium', 'Energy costs partly passed through; diversified revenues'),
    ('Geopolitical', 'Russia / Ukraine',         'Prolonged war keeps Eastern European exposure elevated',                'High',   'Low',    'ISP exited Russia; residual VBI/Serbia exposure is minimal'),
    ('Execution',    "Piano d'Impresa Delivery", 'Management misses fee income and CIR targets from 2025-2028 plan',      'Medium', 'Medium', 'Strong track record — 2022-2025 Plan beat targets'),
    ('Market',       'AUM / Eurizon Outflows',   'Equity market decline triggers AUM redemptions, reducing fee income',   'Medium', 'Medium', 'Sticky retail AUM; bancassurance offsets equity outflows'),
    ('Execution',    'Cyber / IT Risk',          'Cyber incident or IT failure disrupts operations or client data',        'Low',    'High',   'EUR 1.7B annual IT investment; BCM and DR plans in place'),
]

score_map = {
    ('High','High'):'CRITICAL', ('High','Medium'):'HIGH',   ('High','Low'):'MEDIUM',
    ('Medium','High'):'HIGH',   ('Medium','Medium'):'MEDIUM',('Medium','Low'):'LOW',
    ('Low','High'):'MEDIUM',    ('Low','Medium'):'LOW',     ('Low','Low'):'LOW',
}
color_map = {'CRITICAL':'D62728', 'HIGH':'FF7F0E', 'MEDIUM':'FFDD57', 'LOW':'2CA02C'}

print('=' * 100)
print('  RISK REGISTER  |  INTESA SANPAOLO (ISP.MI)')
print('=' * 100)
print(f'  {"#":<3} {"Category":<14} {"Risk":<28} {"Prob":>5} {"Impact":>7} {"Score":<10} Mitigation')
print(f'  {"-"*97}')
for i, (cat, name, desc, prob, impact, mit) in enumerate(risks, 1):
    score = score_map.get((prob, impact), 'MEDIUM')
    print(f'  {i:<3} {cat:<14} {name:<28} {prob:>5} {impact:>7} {score:<10} {mit[:55]}')

# Export to Excel
output_path = 'ISP_Analysis.xlsx'
sheet_name  = 'Risk Register'
wb = load_workbook(output_path)
if sheet_name in wb.sheetnames:
    del wb[sheet_name]
ws = wb.create_sheet(sheet_name)

DARK = '1A1A2E'; WHITE = 'FFFFFF'
ws.merge_cells('A1:G1')
t = ws.cell(row=1, column=1, value='ISP.MI — Risk Register  |  Analyst: Ali Akbar  |  May 2026')
t.fill = PatternFill('solid', fgColor=DARK)
t.font = Font(bold=True, size=13, color=WHITE)
t.alignment = Alignment(horizontal='center')
ws.row_dimensions[1].height = 26

headers = ['#', 'Category', 'Risk Name', 'Probability', 'Impact', 'Score', 'Mitigation']
for ci, h in enumerate(headers, 1):
    c = ws.cell(row=3, column=ci, value=h)
    c.fill = PatternFill('solid', fgColor='333333')
    c.font = Font(bold=True, color=WHITE, size=10)
    c.alignment = Alignment(horizontal='center')

for ri, (cat, name, desc, prob, impact, mit) in enumerate(risks, 4):
    score  = score_map.get((prob, impact), 'MEDIUM')
    s_col  = color_map[score]
    bg     = 'F5F5F5' if ri % 2 == 0 else 'FFFFFF'
    for ci, val in enumerate([ri-3, cat, name, prob, impact, score, mit], 1):
        c = ws.cell(row=ri, column=ci, value=val)
        c.fill = PatternFill('solid', fgColor=s_col if ci == 6 else bg)
        c.font = Font(bold=(ci==6), color=(WHITE if ci==6 else '000000'), size=9)
        c.alignment = Alignment(horizontal='center' if ci != 7 else 'left', wrap_text=True)

col_widths = [4, 13, 24, 11, 9, 10, 55]
for ci, cw in enumerate(col_widths, 1):
    ws.column_dimensions[get_column_letter(ci)].width = cw

wb.save(output_path)
print(f'\nRisk Register saved to {output_path}')


In [ ]:
# ============================================================
# STEP 9 — INVESTMENT THESIS  |  ISP.MI
# Rating, price target, catalysts, risks, written thesis
# Analyst: Ali Akbar
# ============================================================

from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

rating      = 'HOLD'
pt_base     = round(np.mean([v[1] for v in valuation.values()]), 2)
pt_bull     = round(max(v[2] for v in valuation.values()), 2)
pt_bear     = round(min(v[0] for v in valuation.values()), 2)
upside_base = round((pt_base  - current_price) / current_price * 100, 1)
upside_bull = round((pt_bull  - current_price) / current_price * 100, 1)
upside_bear = round((pt_bear  - current_price) / current_price * 100, 1)
div_yield   = round(dps_fwd / current_price * 100, 1)

SEP = '=' * 72
print(SEP)
print(f'  INVESTMENT THESIS  |  INTESA SANPAOLO (ISP.MI)')
print(SEP)
print(f'  Rating        :  {rating}')
print(f'  Price Target  :  EUR {pt_base:.2f}  (12-month, base case)')
print(f'  Bull Case     :  EUR {pt_bull:.2f}  ({upside_bull:+.1f}%)')
print(f'  Bear Case     :  EUR {pt_bear:.2f}  ({upside_bear:+.1f}%)')
print(f'  Current Price :  EUR {current_price:.2f}')
print(f'  Upside (Base) :  {upside_base:+.1f}%')
print(f'  Dividend Yield:  {div_yield:.1f}%  (DPS EUR {dps_fwd:.3f})')

catalysts = [
    ('Fee income acceleration',
     f'Eurizon AUM recovery + bancassurance growth +5-7%/yr (Piano 2025-28). '
     f'Commissioni growing to EUR {results["Bull"].loc[2028,"Fee Income (EUR B)"]:.1f}B by 2028 in bull case.'),
    ('Dividend yield floor',
     f'DPS EUR {dps_fwd:.3f} on EUR {current_price:.2f} = {div_yield:.1f}% yield — '
     f'among highest in European banking. Provides strong downside cushion.'),
    ('Capital optionality',
     f'CET1 ~13.5%, well above regulatory minimum. Surplus capital available '
     f'for additional buybacks or special dividend — not priced in.'),
]

print(f'\n  BULL CATALYSTS')
for i, (title, body) in enumerate(catalysts, 1):
    print(f'\n  {i}. {title}')
    words = body.split(); line = '     '
    for w in words:
        if len(line)+len(w) > 70: print(line); line = '     ' + w + ' '
        else: line += w + ' '
    print(line)

key_risks = [
    ('NIM headwind',
     f'Base NIM compresses from {base_nim*100:.2f}% to '
     f'{results["Base"].loc[2028,"NIM (%)"]:.2f}% by 2028 as ECB cuts rates.'),
    ('Italian credit cycle',
     'Employment -0.1% trend leads NPL deterioration by 6-12 months. '
     'CoR of 55bps in bear case cuts net income by ~EUR 1.5B.'),
    ('Geopolitical / Alexbank',
     'Middle East escalation: energy inflation + Alexbank EGP exposure.'),
]

print(f'\n  KEY RISKS')
for i, (title, body) in enumerate(key_risks, 1):
    print(f'\n  {i}. {title}')
    words = body.split(); line = '     '
    for w in words:
        if len(line)+len(w) > 70: print(line); line = '     ' + w + ' '
        else: line += w + ' '
    print(line)

summary = (
    f'Intesa Sanpaolo is Europe most profitable large bank with ROTE of '
    f'{rote*100:.1f}% and a {div_yield:.1f}% dividend yield that provides a '
    f'compelling total return profile. The six-method valuation consensus of '
    f'EUR {pt_base} implies {upside_base:+.1f}% capital upside plus {div_yield:.1f}% '
    f'yield = ~{upside_base+div_yield:.1f}% total return. We rate ISP HOLD: '
    f'fairly valued but a high-quality compounder with excellent capital returns.'
)
print(f'\n  INVESTMENT SUMMARY')
words = summary.split(); line = '  '
for w in words:
    if len(line)+len(w) > 72: print(line); line = '  ' + w + ' '
    else: line += w + ' '
print(line)

# Export Investment Thesis to Excel
output_path = 'ISP_Analysis.xlsx'
sheet_name  = 'Investment Thesis'
wb = load_workbook(output_path)
if sheet_name in wb.sheetnames:
    del wb[sheet_name]
ws = wb.create_sheet(sheet_name)

DARK = '1A1A2E'; GREEN = '007236'; WHITE = 'FFFFFF'

ws.merge_cells('A1:D1')
t = ws.cell(row=1, column=1, value='ISP.MI — Investment Thesis  |  Analyst: Ali Akbar  |  May 2026')
t.fill = PatternFill('solid', fgColor=DARK)
t.font = Font(bold=True, size=13, color=WHITE)
t.alignment = Alignment(horizontal='center')
ws.row_dimensions[1].height = 26

summary_data = [
    ('Rating', rating),
    ('Price Target (Base)', f'EUR {pt_base}'),
    ('Price Target (Bull)', f'EUR {pt_bull}  ({upside_bull:+.1f}%)'),
    ('Price Target (Bear)', f'EUR {pt_bear}  ({upside_bear:+.1f}%)'),
    ('Current Price', f'EUR {current_price:.2f}'),
    ('Upside (Base)', f'{upside_base:+.1f}%'),
    ('Dividend Yield', f'{div_yield:.1f}%'),
    ('Est. Total Return', f'{upside_base+div_yield:.1f}%'),
]
for ri, (label, value) in enumerate(summary_data, 3):
    bg = 'F0FFF0' if label == 'Rating' else ('F5F5F5' if ri % 2 == 0 else 'FFFFFF')
    c1 = ws.cell(row=ri, column=1, value=label)
    c1.fill = PatternFill('solid', fgColor=bg)
    c1.font = Font(bold=True, size=10)
    c2 = ws.cell(row=ri, column=2, value=value)
    c2.fill = PatternFill('solid', fgColor=bg)
    c2.font = Font(bold=(label=='Rating'), size=10,
                   color=('007236' if label=='Rating' else '000000'))

ws.column_dimensions['A'].width = 28
ws.column_dimensions['B'].width = 28

wb.save(output_path)
print(f'\nInvestment Thesis saved to {output_path}')


In [1]:
# ============================================================
# STEP 10 — LINKEDIN CHARTS  |  ISP.MI  (Dark Theme)
# 5 high-res charts for LinkedIn
# Analyst: Ali Akbar
# ============================================================

import os

OUT_DIR = '.'

# Dark theme palette
BG    = '#0D1117'
BG2   = '#161B22'
GRID  = '#21262D'
BLUE  = '#58A6FF'
GREEN = '#3FB950'
RED   = '#F85149'
GOLD  = '#E3B341'
WHITE = '#E6EDF3'
GREY  = '#8B949E'
TEAL  = '#39D5B0'
CREDIT = 'Ali Akbar  |  ISP.MI Equity Research  |  May 2026'

def base_fig(w=12, h=7):
    fig, ax = plt.subplots(figsize=(w, h), facecolor=BG)
    ax.set_facecolor(BG2)
    ax.tick_params(colors=GREY, labelsize=10)
    ax.spines[:].set_color(GRID)
    ax.xaxis.label.set_color(GREY)
    ax.yaxis.label.set_color(GREY)
    return fig, ax

def add_credit(fig):
    fig.text(0.99, 0.01, CREDIT, ha='right', va='bottom',
             fontsize=8, color=GREY, style='italic')

def save_chart(fig, fname):
    path = os.path.join(OUT_DIR, fname)
    fig.savefig(path, dpi=150, bbox_inches='tight', facecolor=BG)
    plt.close(fig)
    print(f'  Saved: {fname}')

years = ['2022', '2023', '2024', '2025']

# ---- Chart 1: Revenue & Net Income Growth ----------------------
rev = [19.84, 22.35, 24.60, 27.11]
ni  = [4.35,  7.72,  8.35,  9.32]
fig, ax = base_fig()
x = np.arange(len(years)); bw = 0.36
b1 = ax.bar(x-bw/2, rev, bw, color=BLUE,  alpha=0.85, label='Total Revenue', zorder=3)
b2 = ax.bar(x+bw/2, ni,  bw, color=GREEN, alpha=0.85, label='Net Income',    zorder=3)
for b, v in zip(b1, rev): ax.text(b.get_x()+b.get_width()/2, v+0.3, f'EUR {v:.1f}B',  ha='center', color=WHITE, fontsize=10, fontweight='bold')
for b, v in zip(b2, ni):  ax.text(b.get_x()+b.get_width()/2, v+0.3, f'EUR {v:.2f}B', ha='center', color=WHITE, fontsize=10, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(years, color=WHITE, fontsize=12)
ax.set_ylabel('EUR Billions', color=GREY, fontsize=11)
ax.yaxis.grid(True, color=GRID, zorder=0); ax.set_axisbelow(True)
ax.legend(facecolor=BG, edgecolor=GRID, labelcolor=WHITE, fontsize=11)
ax.set_title(f'Intesa Sanpaolo — Revenue & Net Income Growth\nRevenue +{(rev[-1]/rev[0]-1)*100:.0f}%  |  Net Income +{(ni[-1]/ni[0]-1)*100:.0f}%  |  2022-2025',
             color=WHITE, fontsize=14, fontweight='bold', pad=18)
fig.tight_layout(rect=[0,0.04,1,1]); add_credit(fig)
save_chart(fig, 'linkedin_chart1_revenue_growth.png')

# ---- Chart 2: KPI Evolution ------------------------------------
nim_v  = [1.18, 1.53, 1.73, 1.80]
rote_v = [9.7,  16.0, 16.1, 16.9]
cir_v  = [50.1, 47.2, 45.3, 44.0]
fig, ax1 = plt.subplots(figsize=(12,7), facecolor=BG)
ax1.set_facecolor(BG2); ax2 = ax1.twinx(); ax2.set_facecolor(BG2)
l1, = ax1.plot(years, rote_v, 'o-',  color=GREEN, lw=2.5, ms=8, label='ROTE %')
l2, = ax1.plot(years, cir_v,  's--', color=RED,   lw=2.5, ms=8, label='CIR %')
l3, = ax2.plot(years, nim_v,  '^-',  color=GOLD,  lw=2.5, ms=8, label='NIM %')
for y, v in zip(years, rote_v): ax1.annotate(f'{v:.1f}%', (y,v), xytext=(0,10), textcoords='offset points', ha='center', color=GREEN, fontsize=10, fontweight='bold')
for y, v in zip(years, cir_v):  ax1.annotate(f'{v:.1f}%', (y,v), xytext=(0,-16), textcoords='offset points', ha='center', color=RED,   fontsize=10, fontweight='bold')
for y, v in zip(years, nim_v):  ax2.annotate(f'{v:.2f}%', (y,v), xytext=(0,10), textcoords='offset points', ha='center', color=GOLD,  fontsize=10, fontweight='bold')
ax1.axhline(10.1, color=BLUE, lw=1.5, ls=':', alpha=0.7)
ax1.text(years[-1], 10.6, 'Cost of Equity (Ke) 10.1%', color=BLUE, fontsize=9, ha='right')
ax1.set_xticks(range(len(years))); ax1.set_xticklabels(years, color=WHITE, fontsize=12)
for sp in [ax1, ax2]: sp.tick_params(colors=GREY); sp.spines[:].set_color(GRID)
ax1.set_ylabel('ROTE / CIR (%)', color=GREY, fontsize=11)
ax2.set_ylabel('NIM (%)', color=GOLD, fontsize=11)
ax1.yaxis.grid(True, color=GRID)
ax1.legend([l1,l2,l3], ['ROTE %','CIR %','NIM %'], facecolor=BG, edgecolor=GRID, labelcolor=WHITE, fontsize=11, loc='upper left')
ax1.set_title('Intesa Sanpaolo — Key Performance Indicators 2022-2025\nROTE well above Cost of Equity  |  CIR improving  |  NIM at cycle peak',
              color=WHITE, fontsize=14, fontweight='bold', pad=18)
fig.tight_layout(rect=[0,0.04,1,1]); add_credit(fig)
save_chart(fig, 'linkedin_chart2_kpi_evolution.png')

# ---- Chart 3: Scenario Projections -----------------------------
all_yrs = [2025, 2026, 2027, 2028]
base_ni = [9.32,  9.54, 10.04, 10.53]
bull_ni = [9.32, 10.49, 11.59, 12.89]
bear_ni = [9.32,  8.03,  7.62,  7.46]
fig, ax = base_fig()
ax.fill_between(all_yrs, bear_ni, bull_ni, alpha=0.12, color=BLUE, label='Bull-Bear range')
ax.plot(all_yrs, bull_ni, 'o-', color=GREEN, lw=2.5, ms=8, label='Bull  EUR 12.9B (+38%)')
ax.plot(all_yrs, base_ni, 's-', color=BLUE,  lw=2.5, ms=8, label='Base  EUR 10.5B (+13%)')
ax.plot(all_yrs, bear_ni, '^-', color=RED,   lw=2.5, ms=8, label='Bear  EUR 7.5B  (-20%)')
for y,b,ba,br in zip(all_yrs[1:], bull_ni[1:], base_ni[1:], bear_ni[1:]):
    ax.annotate(f'{b:.1f}',  (y,b),  xytext=(0, 9), textcoords='offset points', ha='center', color=GREEN, fontsize=9.5, fontweight='bold')
    ax.annotate(f'{ba:.1f}', (y,ba), xytext=(0,-15), textcoords='offset points', ha='center', color=BLUE,  fontsize=9.5, fontweight='bold')
    ax.annotate(f'{br:.1f}', (y,br), xytext=(0,-15), textcoords='offset points', ha='center', color=RED,   fontsize=9.5, fontweight='bold')
ax.set_xticks(all_yrs); ax.set_xticklabels([str(y) for y in all_yrs], color=WHITE, fontsize=12)
ax.set_ylabel('Net Income (EUR Billions)', color=GREY, fontsize=11)
ax.yaxis.grid(True, color=GRID, zorder=0); ax.set_axisbelow(True)
ax.legend(facecolor=BG, edgecolor=GRID, labelcolor=WHITE, fontsize=11)
ax.set_title('Intesa Sanpaolo — Forward Net Income 2025-2028  (3 Scenarios)\nBase: +13%  |  Bull: +38%  |  Bear: -20%  |  Tax: 27%  |  Analyst: Ali Akbar',
             color=WHITE, fontsize=14, fontweight='bold', pad=18)
fig.tight_layout(rect=[0,0.04,1,1]); add_credit(fig)
save_chart(fig, 'linkedin_chart3_scenario_projections.png')

# ---- Chart 4: Peer Comparison ----------------------------------
peer_data = {
    'ISP.MI':  (1.42, 16.9, 8.9,  GOLD),
    'UCG.MI':  (1.12, 16.3, 7.1,  BLUE),
    'INGA.AS': (0.89, 13.7, 8.5,  TEAL),
    'SAN.MC':  (0.82, 12.0, 5.9,  GREEN),
    'BNP.PA':  (0.65, 10.1, 6.4,  GREY),
    'GLE.PA':  (0.55,  8.9, 7.2,  RED),
}
fig, ax = base_fig()
ax.axhline(10.1, color=BLUE, lw=1.2, ls=':', alpha=0.5)
ax.text(1.55, 10.5, 'Cost of Equity 10.1%', color=BLUE, fontsize=9, ha='center')
for ticker, (pbv, rote_p, divy, col) in peer_data.items():
    ax.scatter(pbv, rote_p, s=divy*120, color=col, alpha=0.80, zorder=5, edgecolors='white', lw=0.8)
    weight = 'bold' if ticker=='ISP.MI' else 'normal'
    fsize  = 12    if ticker=='ISP.MI' else 10
    ax.annotate(f'{ticker}\n{rote_p:.1f}% ROTE  |  {pbv:.2f}x P/BV',
                (pbv, rote_p), xytext=(pbv+0.05, rote_p+0.5),
                color=col, fontsize=fsize, fontweight=weight,
                arrowprops=dict(arrowstyle='-', color=col, lw=0.8))
ax.set_xlabel('P/BV Multiple (x)', color=GREY, fontsize=12)
ax.set_ylabel('ROTE (%)', color=GREY, fontsize=12)
ax.yaxis.grid(True, color=GRID); ax.xaxis.grid(True, color=GRID)
ax.set_axisbelow(True); ax.set_xlim(0.35, 1.75); ax.set_ylim(5, 21)
ax.tick_params(colors=GREY)
ax.set_title('European Bank Peer Comparison — ROTE vs P/BV Multiple\nBubble size = Dividend Yield  |  ISP: best ROTE in class, attractive valuation',
             color=WHITE, fontsize=14, fontweight='bold', pad=18)
fig.tight_layout(rect=[0,0.04,1,1]); add_credit(fig)
save_chart(fig, 'linkedin_chart4_peer_comparison.png')

# ---- Chart 5: Valuation Football Field -------------------------
ff_methods = ['Peer Comps', 'SOTP', 'Excess Return', 'DDM', 'P/E Forward', 'P/BV']
ff_lows    = [3.09, 3.91, 4.20, 4.47, 4.90, 3.81]
ff_highs   = [3.78, 4.78, 5.36, 6.09, 6.64, 5.09]
ff_bases   = [3.43, 4.35, 4.78, 5.32, 5.77, 4.46]
ff_colors  = [TEAL, GREEN, BLUE, GOLD, RED, GREY]
fig, ax = base_fig()
for i, (lo, hi, base, col) in enumerate(zip(ff_lows, ff_highs, ff_bases, ff_colors)):
    ax.barh(i, hi-lo, left=lo, height=0.55, color=col, alpha=0.30, zorder=3)
    ax.barh(i, 0.04, left=base-0.02, height=0.55, color=col, alpha=1.0, zorder=4)
    ax.text(lo-0.08, i, f'{lo:.2f}', va='center', ha='right', color=col, fontsize=10, fontweight='bold')
    ax.text(hi+0.08, i, f'{hi:.2f}', va='center', ha='left',  color=col, fontsize=10, fontweight='bold')
    ax.text(base, i+0.35, f'{base:.2f}', va='bottom', ha='center', color=WHITE, fontsize=9)
ax.axvline(4.52, color=RED,  lw=2,   ls='--', zorder=5, label='Current Price  EUR 4.52')
ax.axvline(4.99, color=GOLD, lw=2.5, ls='-',  zorder=5, label='Target Price   EUR 4.99')
ax.set_yticks(range(len(ff_methods)))
ax.set_yticklabels(ff_methods, color=WHITE, fontsize=11)
ax.set_xlabel('Share Price (EUR)', color=GREY, fontsize=11)
ax.xaxis.grid(True, color=GRID, zorder=0); ax.set_axisbelow(True)
ax.set_xlim(2.5, 7.2); ax.tick_params(colors=GREY)
ax.legend(facecolor=BG, edgecolor=GRID, labelcolor=WHITE, fontsize=11, loc='lower right')
ax.set_title('Intesa Sanpaolo — Valuation Football Field  (6 Methods)\nTarget: EUR 4.99  |  Current: EUR 4.52  |  HOLD  |  Upside: +10.4%',
             color=WHITE, fontsize=14, fontweight='bold', pad=18)
fig.tight_layout(rect=[0,0.04,1,1]); add_credit(fig)
save_chart(fig, 'linkedin_chart5_football_field.png')

print()
print('=' * 60)
print('  ALL 5 LINKEDIN CHARTS SAVED')
print('=' * 60)
print('  1. linkedin_chart1_revenue_growth.png')
print('  2. linkedin_chart2_kpi_evolution.png')
print('  3. linkedin_chart3_scenario_projections.png')
print('  4. linkedin_chart4_peer_comparison.png')
print('  5. linkedin_chart5_football_field.png')
print()
print('  LinkedIn tip: post as a 5-image carousel')
print('  Caption: ISP.MI deep-dive | HOLD | EUR 4.99 PT | +19% total return')
print('  Tags: #EquityResearch #InvestmentBanking #ISP #Finanza #Banking')


NameError: name 'plt' is not defined